# Week 15, Day 4: Continuous distributions, the Gaussian and the central limit theorem

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w015/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Three ways to get one probability

For exponential intervals at 20 Hz, compute P(ISI less than 10 ms) three ways: as the fraction of 100,000 simulated intervals, by integrating `exp_pdf` from 0 to 0.01 with the trapezoid rule on 1,001 points, and from the CDF 1 − exp(−rt).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2008)
r = 20
isi = rng.exponential(1 / r, 100000)

def exp_pdf(t, r):
    return ...

t = np.linspace(0, 0.01, 1001)
p_empirical = ...
p_integral = ...                 # np.trapezoid(y, t), or np.trapz in older numpy
p_formula = ...
print(f"empirical {p_empirical:.4f}, integral {p_integral:.6f}, formula {p_formula:.6f}")
plt.hist(isi, 100, density=True, alpha=0.5, label="histogram (density)"); tt = np.linspace(0, 0.3, 300)
plt.plot(tt, exp_pdf(tt, r), label="exponential density"); plt.xlabel("interval (s)"); plt.ylabel("density (per second)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_three():
    "the empirical fraction, the integral of the density and the CDF formula agree for P(ISI less than 10 ms)"
    assert abs(p_integral - p_formula) < 1e-6 and abs(p_empirical - p_formula) < 0.01

def test_density_units():
    "the density at t = 0 is the rate, 20 per second, greater than 1"
    assert abs(exp_pdf(0.0, 20) - 20) < 1e-12

test_three()
test_density_units()
print("All checks passed")

### Exercise: Standardise and look up

Write `Phi(z)` with `math.erf`. Check the 68-95-99.7 rule on 200,000 standard normal draws (store the three fractions in `within`). Then, for a membrane potential that is Gaussian with mean −58 mV and sd 4 mV, compute `p_cross` = P(V above −50 mV).

In [ ]:
from math import erf, sqrt
def Phi(z):
    return ...

z = rng.normal(size=200000)
within = [np.mean(np.abs(z) < k) for k in (1, 2, 3)]
p_cross = ...
print("within 1, 2, 3 sd:", np.round(within, 4), f"  P(V > -50 mV) = {p_cross:.4f}")

In [ ]:
import numpy as np
def test_phi():
    "Phi matches known values"
    assert abs(Phi(0) - 0.5) < 1e-12 and abs(Phi(1.96) - 0.975) < 1e-3

def test_rule():
    "the empirical fractions within 1, 2 and 3 sd are close to 0.683, 0.954 and 0.997"
    assert np.allclose(within, [0.683, 0.954, 0.997], atol=0.005)

def test_threshold():
    "the probability of crossing threshold matches the Gaussian tail"
    assert abs(p_cross - (1 - Phi((-50 - (-58)) / 4))) < 1e-12

test_phi()
test_rule()
test_threshold()
print("All checks passed")

### Exercise: A membrane potential from 2,000 synapses

Each of 1,600 excitatory and 400 inhibitory synapses fires in a 10 ms window with probability 0.05. Excitatory potentials are exponential with mean 0.5 mV, inhibitory ones exponential with mean −1 mV. Simulate 20,000 windows, predict the mean and variance of the summed potential from the single-synapse moments, and compare the skewness of one synapse with that of the sum.

In [ ]:
nE, nI, p, aE, aI, W = 1600, 400, 0.05, 0.5, 1.0, 20000

def one_synapse_moments(p, a):
    # contribution = (Bernoulli(p)) * (exponential with mean a): mean p a, E[X^2] = p * 2 a^2
    mean = ...
    var = ...
    return mean, var

mE, vE = one_synapse_moments(p, aE)
mI, vI = one_synapse_moments(p, aI)
mean_pred = nE * mE - nI * mI
var_pred = ...

kE = rng.binomial(nE, p, W); kI = rng.binomial(nI, p, W)            # how many of each fire
V = np.array([rng.exponential(aE, ke).sum() - rng.exponential(aI, ki).sum() for ke, ki in zip(kE, kI)])

def skewness(x): return np.mean(((x - x.mean()) / x.std()) ** 3)
single = (rng.random(200000) < p) * rng.exponential(aE, 200000)
skew_single, skew_sum = skewness(single), skewness(V)
p_tail_sim = np.mean(V > mean_pred + 2 * np.sqrt(var_pred))
p_tail_gauss = 1 - Phi(2)
print(f"mean {V.mean():.3f} (predicted {mean_pred:.3f}),  variance {V.var():.3f} (predicted {var_pred:.3f})")
print(f"skewness: one synapse {skew_single:.1f}, sum {skew_sum:.3f};  P(above mean + 2 sd): {p_tail_sim:.4f} vs Gaussian {p_tail_gauss:.4f}")
plt.hist(V, 80, density=True, alpha=0.5); x = np.linspace(V.min(), V.max(), 200)
plt.plot(x, np.exp(-(x - mean_pred) ** 2 / (2 * var_pred)) / np.sqrt(2 * np.pi * var_pred)); plt.xlabel("summed potential (mV)"); plt.show()

In [ ]:
import numpy as np
def test_moments():
    "the simulated mean and variance match the predictions from linearity and independence to within 3%"
    assert abs(V.mean() - mean_pred) / abs(mean_pred) < 0.03 and abs(V.var() - var_pred) / var_pred < 0.03

def test_skew():
    "one synapse's contribution is very skewed, the sum nearly symmetric"
    assert skew_single > 5 and abs(skew_sum) < 0.3

def test_tail():
    "the Gaussian with the predicted mean and variance gets P(V above mean + 2 sd) right to within 0.005"
    assert abs(p_tail_sim - p_tail_gauss) < 0.005

test_moments()
test_skew()
test_tail()
print("All checks passed")

### Exercise: An average that never settles

For n = 10 and n = 1000, take 5,000 sample means of standard Cauchy draws and of standard normal draws, and store the interquartile range of the means (75th minus 25th percentile) in `iqr_cauchy` and `iqr_normal`.

In [ ]:
def iqr(x): return np.percentile(x, 75) - np.percentile(x, 25)
iqr_cauchy = [iqr(rng.standard_cauchy((5000, n)).mean(axis=1)) for n in (10, 1000)]
iqr_normal = ...
print("Cauchy IQR of means for n = 10, 1000:", np.round(iqr_cauchy, 3), "   normal:", np.round(iqr_normal, 3))

In [ ]:
import numpy as np
def test_cauchy():
    "the interquartile range of Cauchy sample means does not shrink from n = 10 to n = 1000"
    assert iqr_cauchy[1] > 0.7 * iqr_cauchy[0]

def test_normal():
    "for normal draws it shrinks about tenfold"
    assert 7 < iqr_normal[0] / iqr_normal[1] < 13

test_cauchy()
test_normal()
print("All checks passed")